# Appendix C: Review of Group Theory

**Source Span:** John M. Lee, *Introduction to Topological Manifolds*, Second Edition, Appendix C, printed pp. 401-406; PDF pp. 419-424.

## Appendix Goal

This appendix is a compact algebra toolkit for the topology chapters. The goal of this notebook is not to replace an algebra course. It is to make the group-theoretic moves that appear later in the book visible, checkable, and ready to use: multiplication, identity, inverses, generated subgroups, products, homomorphisms, kernels, images, conjugation, normality, quotient groups, presentations, cyclic groups, and group actions.

Groups enter topology because paths and symmetries can be multiplied. A loop followed by another loop behaves like a product. A covering transformation behaves like an automorphism. A surface presentation gives generators and relations for a fundamental group. A group action turns a symmetry group into an organized way to move points of a space. The same few algebraic invariants keep reappearing: which elements collapse to the identity, which subgroups survive conjugation, which cosets become points of a quotient, and which data determine a homomorphism.

The running finite model is the symmetry group of an equilateral triangle, usually written `S_3` or `D_3`. It is small enough to enumerate but rich enough to show noncommutativity, conjugation, a normal subgroup, a quotient, a presentation, and an action. Treat it as a microscope: if a proposed construction fails here, it is probably not safe in a fundamental-group calculation either.

## Computational Translation Guide

| Book concept | Computational representation | What to inspect |
| --- | --- | --- |
| group operation | a finite multiplication function on permutations | closure, associativity, identity, inverses |
| generated subgroup | closure under selected generators and inverses | whether repeated products stop at a smaller set or fill the group |
| direct product | componentwise modular arithmetic | product size and coordinatewise multiplication |
| homomorphism | a function respecting multiplication | kernel, image, and whether products are preserved |
| conjugation and normality | `g*k*g^{-1}` for all `g` | whether a subgroup is fixed as a set |
| quotient group | cosets with induced multiplication | representative independence |
| presentation | generators plus relations evaluated in a model | whether the relators really become identity |
| group action | permutations of a set of vertices | orbit, stabilizer, and orbit-stabilizer count |

## Library Routing

This chapter is algebraic, so the most useful geometry is finite and combinatorial. `networkx` is used for Cayley and dependency graphs because products and proof dependencies are graph-shaped data. `matplotlib` is used for durable static PNG diagrams of Cayley graphs, cosets, and action fibers. `plotly` is used once for an interactive product grid, where hovering over a point is more useful than a static table. `pandas` stores multiplication, quotient, and lab tables. `numpy` supplies reliable planar coordinates for diagrams. The notebook avoids heavier geometry libraries because no surface, mesh, metric, or persistent-homology computation is needed for this appendix.

## Visual Storyboard

1. Build the triangle symmetry group from two generators and save a Cayley graph. Inspection target: the generator arrows show how six symmetries are reached from `r` and `s`.
2. Collapse `S_3` by the sign homomorphism and show the two cosets of the alternating subgroup. Inspection target: normality is visible as a quotient with a well-defined two-element product.
3. Show `C_2 x C_3` as a product grid and identify it with `C_6`. Inspection target: componentwise products can still generate a single cycle when the factors have coprime orders.
4. Verify a presentation for triangle symmetries and tabulate shortest words. Inspection target: relations are not decorations; they are equations that force many words to name the same element.
5. Visualize the action on triangle vertices through orbit and stabilizer fibers. Inspection target: the size equation `|G| = |orbit| * |stabilizer|` is a count of visible fibers.


In [ ]:
# geometry-setup:v1
# Machine-managed by scripts/update_notebook_setup.py. Do not edit this cell by hand.

from __future__ import annotations

import json as _geometry_json
import os as _geometry_os
from pathlib import Path as _GeometryPath
import sys as _geometry_sys

GEOMETRY_SETUP = _geometry_json.loads(
    r"""
{
  "colab_url": "https://colab.research.google.com/github/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/appendix-c-review-of-group-theory/appendix-c-review-of-group-theory.ipynb",
  "course_dir": "Introduction-to-Topological-Manifolds",
  "course_title": "Introduction to Topological Manifolds",
  "github_url": "https://github.com/Rah-Rah-Mitra/Geometry/blob/main/Introduction-to-Topological-Manifolds/appendix-c-review-of-group-theory/appendix-c-review-of-group-theory.ipynb",
  "jupyterlite": false,
  "marker": "geometry-setup:v1",
  "notebook_kind": "lesson",
  "notebook_path": "Introduction-to-Topological-Manifolds/appendix-c-review-of-group-theory/appendix-c-review-of-group-theory.ipynb",
  "notebook_title": "Appendix C: Review of Group Theory",
  "repository": {
    "branch": "main",
    "name": "Geometry",
    "owner": "Rah-Rah-Mitra",
    "source_url": "https://github.com/Rah-Rah-Mitra/Geometry"
  },
  "requirements": "requirements/topology.txt",
  "runtime_profile": "topology"
}
"""
)


def _geometry_is_colab():
    try:
        import google.colab  # type: ignore  # noqa: F401
        return True
    except Exception:
        return False


def _geometry_is_jupyterlite():
    return _geometry_sys.platform == "emscripten" or "pyodide" in _geometry_sys.modules


def _geometry_add_path(path):
    text = str(path)
    if text not in _geometry_sys.path:
        _geometry_sys.path.insert(0, text)


def _geometry_find_repo_root():
    candidates = []
    env_root = _geometry_os.environ.get("GEOMETRY_REPO_ROOT")
    if env_root:
        candidates.append(_GeometryPath(env_root).expanduser())
    candidates.append(_GeometryPath.cwd())
    for start in candidates:
        start = start.resolve()
        for current in (start, *start.parents):
            if (current / "course-manifest.json").exists() and (
                current / "metadata" / "runtime_profiles.yml"
            ).exists():
                return current
    raise RuntimeError(
        "Could not find the Geometry repository root. Start JupyterLab inside the "
        "Geometry checkout or set GEOMETRY_REPO_ROOT."
    )


def _geometry_run(command):
    import subprocess as _geometry_subprocess

    printable = " ".join(str(part) for part in command)
    print(f"+ {printable}")
    _geometry_subprocess.check_call([str(part) for part in command])


def _geometry_requirement_names(requirements_path, seen=None):
    seen = set() if seen is None else seen
    requirements_path = requirements_path.resolve()
    if requirements_path in seen or not requirements_path.exists():
        return []
    seen.add(requirements_path)
    names = []
    for raw_line in requirements_path.read_text(encoding="utf-8").splitlines():
        line = raw_line.split("#", 1)[0].strip()
        if not line:
            continue
        if line.startswith(("-r ", "--requirement ")):
            _, nested = line.split(maxsplit=1)
            names.extend(_geometry_requirement_names(requirements_path.parent / nested, seen))
            continue
        if line.startswith("-"):
            continue
        name = line
        for separator in ("==", ">=", "<=", "~=", "!=", ">", "<", ";"):
            name = name.split(separator, 1)[0]
        name = name.split("[", 1)[0].strip()
        if name:
            names.append(name)
    return sorted(set(names))


def _geometry_missing_requirements(requirements_path):
    import importlib.metadata as _geometry_metadata

    missing = []
    for name in _geometry_requirement_names(requirements_path):
        try:
            _geometry_metadata.distribution(name)
        except _geometry_metadata.PackageNotFoundError:
            missing.append(name)
    return missing


def _geometry_configured_roots(repo_root):
    course_dir = GEOMETRY_SETUP.get("course_dir")
    course_root = repo_root / course_dir if course_dir else repo_root
    return repo_root, course_root


if _geometry_is_jupyterlite():
    if not GEOMETRY_SETUP["jupyterlite"]:
        raise RuntimeError(
            "This Geometry notebook uses runtime profile "
            f"{GEOMETRY_SETUP['runtime_profile']!r}, which is not enabled for "
            "JupyterLite in course-manifest.json. Open it in Colab or local JupyterLab."
        )
    GEOMETRY_REPO_ROOT = _GeometryPath.cwd()
    GEOMETRY_COURSE_ROOT = (
        GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["course_dir"]
        if GEOMETRY_SETUP.get("course_dir")
        else GEOMETRY_REPO_ROOT
    )
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    if GEOMETRY_COURSE_ROOT.exists():
        _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        "Geometry setup: JupyterLite/Pyodide detected; shell, git, and pip steps "
        "were skipped."
    )
elif _geometry_is_colab():
    repository = GEOMETRY_SETUP["repository"]
    repo_url = repository["source_url"].rstrip("/") + ".git"
    branch = repository["branch"]
    GEOMETRY_REPO_ROOT = _GeometryPath(
        _geometry_os.environ.get("GEOMETRY_REPO_ROOT", "/content/Geometry")
    )
    sparse_paths = ["requirements", "metadata", "scripts", "course-manifest.json", "index.ipynb"]
    if GEOMETRY_SETUP.get("course_dir"):
        sparse_paths.append(GEOMETRY_SETUP["course_dir"])
    if not (GEOMETRY_REPO_ROOT / ".git").exists():
        if GEOMETRY_REPO_ROOT.exists() and any(GEOMETRY_REPO_ROOT.iterdir()):
            raise RuntimeError(
                f"{GEOMETRY_REPO_ROOT} exists but is not a git checkout. "
                "Set GEOMETRY_REPO_ROOT to an empty path or remove the directory."
            )
        _geometry_run(
            [
                "git",
                "clone",
                "--filter=blob:none",
                "--no-checkout",
                "--branch",
                branch,
                repo_url,
                GEOMETRY_REPO_ROOT,
            ]
        )
        _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "init", "--cone"])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "sparse-checkout", "set", *sparse_paths])
    _geometry_run(["git", "-C", GEOMETRY_REPO_ROOT, "checkout", branch])
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-q", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: Colab ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )
else:
    GEOMETRY_REPO_ROOT = _geometry_find_repo_root()
    requirements_path = GEOMETRY_REPO_ROOT / GEOMETRY_SETUP["requirements"]
    missing = _geometry_missing_requirements(requirements_path)
    skip_install = _geometry_os.environ.get("GEOMETRY_SKIP_INSTALL") == "1"
    if missing and skip_install:
        print(
            "Geometry setup: GEOMETRY_SKIP_INSTALL=1, so missing profile packages "
            f"were not installed: {', '.join(missing)}"
        )
    elif missing:
        print(
            "Geometry setup: installing missing profile packages from "
            f"{requirements_path.relative_to(GEOMETRY_REPO_ROOT)}: {', '.join(missing)}"
        )
        _geometry_run([_geometry_sys.executable, "-m", "pip", "install", "-r", requirements_path])
    GEOMETRY_REPO_ROOT, GEOMETRY_COURSE_ROOT = _geometry_configured_roots(GEOMETRY_REPO_ROOT)
    _geometry_os.chdir(GEOMETRY_COURSE_ROOT if GEOMETRY_COURSE_ROOT.exists() else GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_REPO_ROOT)
    _geometry_add_path(GEOMETRY_COURSE_ROOT)
    GEOMETRY_RUNTIME_PROFILE = GEOMETRY_SETUP["runtime_profile"]
    print(
        f"Geometry setup: local checkout ready at {_GeometryPath.cwd()} "
        f"with profile {GEOMETRY_RUNTIME_PROFILE!r}."
    )


In [ ]:
from pathlib import Path
import itertools
import json
import math
import sys

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import plotly.graph_objects as go


def locate_book_root(start=None):
    current = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'source_map.json').exists():
            return candidate
        nested = candidate / 'Introduction-to-Topological-Manifolds'
        if (nested / 'AGENTS.md').exists() and (nested / 'source_map.json').exists():
            return nested
    raise RuntimeError('Could not locate the Introduction-to-Topological-Manifolds book root')


BOOK_ROOT = locate_book_root()
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import assert_artifacts, chapter_artifact_root, display_artifact, save_csv, save_json, save_matplotlib, save_plotly_html
from utils.validation import image_stats

UNIT_KEY = 'appendix-c-review-of-group-theory'
ARTIFACT_ROOT = chapter_artifact_root(UNIT_KEY, BOOK_ROOT)
FIGURES = ARTIFACT_ROOT / 'figures'
HTML = ARTIFACT_ROOT / 'html'
CHECKS = ARTIFACT_ROOT / 'checks'
TABLES = ARTIFACT_ROOT / 'tables'

print(f'Book root: {BOOK_ROOT.relative_to(BOOK_ROOT.parent)}')
print(f'Artifact root: {ARTIFACT_ROOT.relative_to(BOOK_ROOT)}')


In [ ]:
visual_storyboard = [
    {
        'concept': 'triangle symmetry group',
        'representation': 'Cayley graph for S3 generated by r and s',
        'library': 'networkx + matplotlib',
        'artifact': 'figures/triangle-symmetry-cayley-graph.png',
        'inspection_target': 'follow r and s arrows to see generation and noncommuting products',
        'validation': 'closure, associativity, identity, inverse, and presentation relations',
    },
    {
        'concept': 'homomorphism, kernel, quotient',
        'representation': 'sign map with coset collapse and proof dependency graph',
        'library': 'matplotlib + networkx + pandas',
        'artifact': 'figures/quotient-cosets-sign-map.png',
        'inspection_target': 'see six elements collapse to two cosets and then to C2',
        'validation': 'kernel equals A3, normality, representative-independent quotient product',
    },
    {
        'concept': 'direct products',
        'representation': 'interactive C2 x C3 grid with CRT labels in C6',
        'library': 'plotly',
        'artifact': 'html/product-grid-cyclic-crt.html',
        'inspection_target': 'hover over product elements and trace the generator (1,1)',
        'validation': 'CRT map is bijective and respects addition',
    },
    {
        'concept': 'presentations',
        'representation': 'word evaluator and shortest-word table for <r,s | r^3=s^2=1, srs=r^-1>',
        'library': 'pandas',
        'artifact': 'tables/presentation-word-lab.csv',
        'inspection_target': 'different words can evaluate to the same symmetry once relations are imposed',
        'validation': 'all relators evaluate to the identity permutation',
    },
    {
        'concept': 'group actions',
        'representation': 'orbit-stabilizer fibers for the action on triangle vertices',
        'library': 'matplotlib + pandas',
        'artifact': 'figures/orbit-stabilizer-triangle-action.png',
        'inspection_target': 'the stabilizer fibers partition the group over an orbit',
        'validation': '|S3| = |orbit(1)| * |stabilizer(1)|',
    },
]

storyboard_path = save_json(visual_storyboard, CHECKS / 'visual-storyboard.json')
display_artifact(storyboard_path)


## 1. Groups as Multiplication You Can Inspect

A group is a set with a product, an identity element, and inverses, with one crucial compatibility condition: associativity. In topology the product is often path concatenation or composition of symmetries, and associativity is what lets us ignore parentheses when multiplying several classes or transformations. The finite case lets us test the entire definition by enumeration.

The triangle symmetry group has six elements. Let `r` rotate the vertices `1 -> 2 -> 3 -> 1`, and let `s` reflect the triangle while fixing vertex `1`. Every symmetry is a product of these two operations. The relation `r^3 = 1` says that three rotations bring the triangle back. The relation `s^2 = 1` says that reflecting twice does nothing. The relation `srs = r^{-1}` records that a reflection reverses orientation. Those three relations are also a presentation of the group, a compact algebraic description that later chapters use for fundamental groups.

The Cayley graph below is a multiplication dashboard. A blue arrow means multiply by `r`; a red arrow means multiply by `s`. If the graph reaches all six nodes from the identity, the chosen generators really generate the group. If the blue and red arrows do not form a square at every node, the group is not abelian: the products `rs` and `sr` are different operations.


In [ ]:
E = (0, 1, 2)
r = (1, 2, 0)
s = (0, 2, 1)


def compose(p, q):
    return tuple(p[q[i]] for i in range(len(p)))


def inverse(p):
    inv = [None] * len(p)
    for i, image in enumerate(p):
        inv[image] = i
    return tuple(inv)


def power(p, n):
    if n < 0:
        return power(inverse(p), -n)
    out = tuple(range(len(p)))
    for _ in range(n):
        out = compose(p, out)
    return out


def generated_group(generators):
    gens = list(generators)
    gens += [inverse(g) for g in gens]
    seen = {E}
    frontier = [E]
    while frontier:
        current = frontier.pop()
        for gen in gens:
            nxt = compose(gen, current)
            if nxt not in seen:
                seen.add(nxt)
                frontier.append(nxt)
    return seen


def cycle_name(p):
    if p == tuple(range(len(p))):
        return '1'
    visited = [False] * len(p)
    cycles = []
    for start in range(len(p)):
        if visited[start] or p[start] == start:
            visited[start] = True
            continue
        cur = start
        cycle = []
        while not visited[cur]:
            visited[cur] = True
            cycle.append(str(cur + 1))
            cur = p[cur]
        cycles.append('(' + ''.join(cycle) + ')')
    return ''.join(cycles)


S3 = sorted(generated_group([r, s]), key=lambda p: (p != E, cycle_name(p)))
name_of = {g: cycle_name(g) for g in S3}

closure_ok = all(compose(a, b) in S3 for a in S3 for b in S3)
associative_ok = all(compose(compose(a, b), c) == compose(a, compose(b, c)) for a in S3 for b in S3 for c in S3)
identity_ok = all(compose(E, g) == g and compose(g, E) == g for g in S3)
inverse_ok = all(compose(g, inverse(g)) == E and compose(inverse(g), g) == E for g in S3)
relations_ok = {
    'r^3=1': power(r, 3) == E,
    's^2=1': power(s, 2) == E,
    'srs=r^-1': compose(compose(s, r), s) == inverse(r),
}

group_check = {
    'group_order': len(S3),
    'elements': [name_of[g] for g in S3],
    'closure': closure_ok,
    'associativity': associative_ok,
    'identity': identity_ok,
    'inverses': inverse_ok,
    'presentation_relations': relations_ok,
    'noncommuting_example': {
        'r_then_s': name_of[compose(s, r)],
        's_then_r': name_of[compose(r, s)],
        'different': compose(s, r) != compose(r, s),
    },
}

table_rows = []
for a in S3:
    row = {'left': name_of[a]}
    for b in S3:
        row[name_of[b]] = name_of[compose(a, b)]
    table_rows.append(row)
multiplication_table_path = save_csv(table_rows, TABLES / 's3-multiplication-table.csv')
group_check_path = save_json(group_check, CHECKS / 'group-axioms-and-presentation.json')

nodes = [name_of[g] for g in S3]
pos = nx.circular_layout(nodes, scale=2.4)
r_edges = [(name_of[g], name_of[compose(r, g)]) for g in S3]
s_edges = [(name_of[g], name_of[compose(s, g)]) for g in S3]

fig, ax = plt.subplots(figsize=(7.5, 6.2))
nx.draw_networkx_nodes(nx.DiGraph(), pos, nodelist=nodes, node_size=1550, node_color='#f7f0d7', edgecolors='#333333', ax=ax)
nx.draw_networkx_labels(nx.DiGraph(), pos, labels={node: node for node in nodes}, font_size=11, ax=ax)
nx.draw_networkx_edges(nx.DiGraph(r_edges), pos, edge_color='#2b6cb0', arrows=True, arrowsize=15, width=2.0, connectionstyle='arc3,rad=0.12', ax=ax)
nx.draw_networkx_edges(nx.DiGraph(s_edges), pos, edge_color='#b8323f', arrows=True, arrowsize=15, width=1.7, connectionstyle='arc3,rad=-0.18', ax=ax)
ax.plot([], [], color='#2b6cb0', linewidth=2, label='multiply by r')
ax.plot([], [], color='#b8323f', linewidth=2, label='multiply by s')
ax.set_title('Cayley graph of triangle symmetries generated by r and s')
ax.legend(loc='upper right', frameon=False)
ax.set_axis_off()
cayley_path = save_matplotlib(fig, FIGURES / 'triangle-symmetry-cayley-graph.png')
plt.close(fig)

display_artifact(cayley_path, width=720)
display_artifact(group_check_path)
pd.DataFrame(table_rows)


## 2. Homomorphisms, Kernels, and Quotients

A homomorphism is the algebraic version of a structure-preserving map. It does not have to remember everything. Its kernel records exactly what it forgets: the elements that are sent to the identity. For a topology example, an induced map on fundamental groups may collapse loops that become null-homotopic after applying a continuous map. The kernel is the algebraic footprint of that collapse.

The sign homomorphism from `S_3` to `C_2` sends rotations to `+1` and reflections to `-1`. Its kernel is the three-element subgroup `A_3` of even permutations. That subgroup is normal, which means conjugation by any symmetry keeps it inside itself. Normality is not a cosmetic condition. It is exactly what makes coset multiplication independent of the representatives chosen. If `K` is not normal, the formula `(gK)(hK) = (gh)K` can change when `g` or `h` is replaced by another member of the same coset.

The first isomorphism theorem says that every homomorphism factors into two conceptual steps: first collapse the kernel, then identify the resulting quotient with the image. In this finite model, `S_3/A_3` is a two-element group, and it is isomorphic to the image of the sign map. The diagram below shows the collapse from six elements to two cosets and then to the two signs. The dependency graph beside it records the proof logic: homomorphism gives a normal kernel; normality gives a quotient; quotient well-definedness lets the descended map exist; trivial descended kernel gives an isomorphism onto the image.


In [ ]:
def sign(p):
    inversions = 0
    for i in range(len(p)):
        for j in range(i + 1, len(p)):
            inversions += int(p[i] > p[j])
    return 1 if inversions % 2 == 0 else -1


A3 = {g for g in S3 if sign(g) == 1}
odd_coset = set(S3) - A3
cosets = {'A3': A3, 'sA3': odd_coset}


def coset_label(g):
    return 'A3' if g in A3 else 'sA3'


normal_A3 = all(compose(compose(g, k), inverse(g)) in A3 for g in S3 for k in A3)
kernel_equals_A3 = {g for g in S3 if sign(g) == 1} == A3
sign_homomorphism = all(sign(compose(a, b)) == sign(a) * sign(b) for a in S3 for b in S3)

quotient_rows = []
well_defined_products = True
for left_name, left_coset in cosets.items():
    row = {'left_coset': left_name, 'left_members': ', '.join(sorted(name_of[g] for g in left_coset))}
    for right_name, right_coset in cosets.items():
        possible = {coset_label(compose(a, b)) for a in left_coset for b in right_coset}
        well_defined_products = well_defined_products and len(possible) == 1
        row[f'{right_name}_times'] = sorted(possible)[0]
        row[f'{right_name}_members'] = ', '.join(sorted(name_of[g] for g in right_coset))
    quotient_rows.append(row)

quotient_table_path = save_csv(quotient_rows, TABLES / 'sign-quotient-cosets.csv')
quotient_check = {
    'sign_is_homomorphism': sign_homomorphism,
    'kernel_equals_A3': kernel_equals_A3,
    'A3_elements': sorted(name_of[g] for g in A3),
    'odd_coset_elements': sorted(name_of[g] for g in odd_coset),
    'A3_is_normal': normal_A3,
    'quotient_product_well_defined': well_defined_products,
    'quotient_order': len(cosets),
    'image_order': len({sign(g) for g in S3}),
    'first_isomorphism_model_holds': all([sign_homomorphism, kernel_equals_A3, normal_A3, well_defined_products, len(cosets) == len({sign(g) for g in S3})]),
}
quotient_check_path = save_json(quotient_check, CHECKS / 'quotient-first-isomorphism.json')

fig, ax = plt.subplots(figsize=(9.0, 5.6))
element_y = np.linspace(0.85, 0.15, len(S3))
element_positions = {g: (0.08, y) for g, y in zip(S3, element_y)}
coset_positions = {'A3': (0.52, 0.68), 'sA3': (0.52, 0.32)}
sign_positions = {1: (0.88, 0.68), -1: (0.88, 0.32)}

for g, (x, y) in element_positions.items():
    c = coset_label(g)
    sx, sy = coset_positions[c]
    ax.plot([x + 0.03, sx - 0.06], [y, sy], color='#999999', linewidth=1.0)
    ax.scatter([x], [y], s=460, color='#f7f0d7', edgecolor='#333333', zorder=3)
    ax.text(x, y, name_of[g], ha='center', va='center', fontsize=9)

for label, members in cosets.items():
    x, y = coset_positions[label]
    color = '#dbeafe' if label == 'A3' else '#fee2e2'
    ax.scatter([x], [y], s=1900, color=color, edgecolor='#333333', zorder=3)
    ax.text(x, y + 0.025, label, ha='center', va='center', fontsize=12, weight='bold')
    ax.text(x, y - 0.055, ', '.join(sorted(name_of[g] for g in members)), ha='center', va='center', fontsize=8)
    sign_value = 1 if label == 'A3' else -1
    tx, ty = sign_positions[sign_value]
    ax.plot([x + 0.08, tx - 0.06], [y, ty], color='#555555', linewidth=1.5)

for sign_value, (x, y) in sign_positions.items():
    ax.scatter([x], [y], s=1050, color='#ecfdf5', edgecolor='#333333', zorder=3)
    ax.text(x, y, '+1' if sign_value == 1 else '-1', ha='center', va='center', fontsize=13, weight='bold')

ax.text(0.08, 0.97, 'elements of S3', ha='center', fontsize=11, weight='bold')
ax.text(0.52, 0.97, 'cosets of A3', ha='center', fontsize=11, weight='bold')
ax.text(0.88, 0.97, 'image in C2', ha='center', fontsize=11, weight='bold')
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_axis_off()
ax.set_title('The sign homomorphism factors through the quotient S3/A3')
quotient_fig_path = save_matplotlib(fig, FIGURES / 'quotient-cosets-sign-map.png')
plt.close(fig)

proof_edges = [
    ('homomorphism f', 'kernel K'),
    ('kernel K', 'K normal'),
    ('K normal', 'quotient G/K'),
    ('quotient G/K', 'descended map'),
    ('K in kernel', 'descended map'),
    ('descended map', 'image Im f'),
    ('trivial descended kernel', 'isomorphism onto image'),
    ('image Im f', 'isomorphism onto image'),
]
proof_graph = nx.DiGraph(proof_edges)
proof_pos = nx.spring_layout(proof_graph, seed=7, k=1.0)
fig, ax = plt.subplots(figsize=(8.5, 5.4))
nx.draw_networkx_nodes(proof_graph, proof_pos, node_size=1700, node_color='#eef2ff', edgecolors='#333333', ax=ax)
nx.draw_networkx_labels(proof_graph, proof_pos, font_size=8, ax=ax)
nx.draw_networkx_edges(proof_graph, proof_pos, arrows=True, arrowsize=16, width=1.6, edge_color='#4b5563', ax=ax)
ax.set_title('Proof scaffold for the first isomorphism theorem')
ax.set_axis_off()
proof_path = save_matplotlib(fig, FIGURES / 'first-isomorphism-proof-dependencies.png')
plt.close(fig)

display_artifact(quotient_fig_path, width=760)
display_artifact(proof_path, width=720)
display_artifact(quotient_check_path)
pd.DataFrame(quotient_rows)


## 3. Products and Direct Sums

The direct product of groups multiplies coordinates independently. In a finite product this is easy to see: an element of `C_2 x C_3` is a pair `(a,b)`, and multiplication is addition modulo `2` in the first coordinate and modulo `3` in the second. The product has `2 * 3 = 6` elements.

A product is not automatically cyclic. The pair of factor orders matters. Because `2` and `3` are coprime, the element `(1,1)` cycles through all six elements of `C_2 x C_3`; this product is isomorphic to `C_6`. The Chinese remainder map shown below labels each pair by the unique residue modulo `6` with those two remainders. The homomorphism check is exact integer arithmetic: adding pairs and then converting to `C_6` gives the same answer as converting each pair and adding in `C_6`.

The direct sum distinction matters for infinite families of abelian groups. A direct product allows infinitely many nonidentity coordinates, while a direct sum keeps only finitely supported tuples. Later, finitely supported chains and cochains behave like direct sums, not arbitrary direct products: a chain is a finite formal combination of simplices.


In [ ]:
C2xC3 = [(a, b) for a in range(2) for b in range(3)]


def add_product(u, v):
    return ((u[0] + v[0]) % 2, (u[1] + v[1]) % 3)


def crt_to_c6(u):
    return (3 * u[0] + 4 * u[1]) % 6


product_homomorphism = all(crt_to_c6(add_product(u, v)) == (crt_to_c6(u) + crt_to_c6(v)) % 6 for u in C2xC3 for v in C2xC3)
product_bijection = sorted(crt_to_c6(u) for u in C2xC3) == list(range(6))
orbit = []
current = (0, 0)
for _ in range(6):
    orbit.append(current)
    current = add_product(current, (1, 1))
generator_has_order_six = current == (0, 0) and len(set(orbit)) == 6

product_check = {
    'group': 'C2 x C3',
    'order': len(C2xC3),
    'crt_map_bijective': product_bijection,
    'crt_map_homomorphism': product_homomorphism,
    'element_(1,1)_has_order_6': generator_has_order_six,
    'orbit_of_(1,1)': [str(u) for u in orbit],
}
product_check_path = save_json(product_check, CHECKS / 'product-crt-cyclic-check.json')

x = [u[1] for u in C2xC3]
y = [u[0] for u in C2xC3]
labels = [f'{u} -> {crt_to_c6(u)} mod 6' for u in C2xC3]
colors = [crt_to_c6(u) for u in C2xC3]
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode='markers+text',
    marker=dict(size=34, color=colors, colorscale='Viridis', line=dict(color='black', width=1)),
    text=[str(crt_to_c6(u)) for u in C2xC3],
    textposition='middle center',
    hovertext=labels,
    hoverinfo='text',
    name='C2 x C3 elements',
))
orbit_x = [u[1] for u in orbit + [orbit[0]]]
orbit_y = [u[0] for u in orbit + [orbit[0]]]
fig.add_trace(go.Scatter(x=orbit_x, y=orbit_y, mode='lines', line=dict(color='firebrick', width=3), hoverinfo='skip', name='add (1,1)'))
fig.update_layout(
    title='Direct product C2 x C3 labeled by the Chinese remainder map to C6',
    xaxis=dict(title='C3 coordinate', tickmode='array', tickvals=[0, 1, 2]),
    yaxis=dict(title='C2 coordinate', tickmode='array', tickvals=[0, 1], range=[-0.35, 1.35]),
    width=780,
    height=430,
    template='plotly_white',
)
product_html_path = save_plotly_html(fig, HTML / 'product-grid-cyclic-crt.html')

display_artifact(product_html_path, width=780, height=470)
display_artifact(product_check_path)
pd.DataFrame({'element': C2xC3, 'crt_label_mod_6': [crt_to_c6(u) for u in C2xC3]})


## 4. Presentations as Executable Relations

A presentation is a promise: start with formal words in generators, then impose specified equations called relations. The triangle group model satisfies the presentation `<r,s | r^3, s^2, srsr>`, equivalently `r^3 = 1`, `s^2 = 1`, and `srs = r^{-1}`. The relation list is short, but it changes the meaning of words dramatically. The words `rs`, `sR`, and longer variants can describe the same symmetry after the relations are enforced.

In topology, presentations are a main output of Van Kampen computations. Generators often come from loops, while relations come from attaching maps or from the way open sets overlap. This is why presentations must be treated as data with checks, not just notation. A word evaluator in a known finite model gives a sanity check: each defining relation should evaluate to the identity, and each element should have a short representative word if the generators really generate the model.

The lab below evaluates words in `r`, `R` for `r^{-1}`, and `s`. Try changing `candidate_word` to another expression. The evaluation does not solve the general word problem for arbitrary presentations; it simply tests this presentation in this concrete permutation model. That limitation is important. Later notebooks can use presentations as exact algebraic descriptions, but computations with arbitrary presentations need additional algorithms or external systems.


In [ ]:
GENERATOR_PERMS = {'r': r, 'R': inverse(r), 's': s}


def eval_word(word):
    out = E
    for letter in word:
        if letter.strip() == '':
            continue
        out = compose(GENERATOR_PERMS[letter], out)
    return out


def shortest_words(generators):
    shortest = {E: ''}
    queue = [E]
    while queue:
        current = queue.pop(0)
        for letter, perm in generators.items():
            nxt = compose(perm, current)
            if nxt not in shortest:
                shortest[nxt] = shortest[current] + letter
                queue.append(nxt)
    return shortest


shortest = shortest_words({'r': r, 's': s})
presentation_rows = [
    {'element': name_of[g], 'permutation_tuple': str(g), 'shortest_word': shortest[g] or 'identity', 'word_length': len(shortest[g])}
    for g in S3
]
presentation_table_path = save_csv(presentation_rows, TABLES / 'presentation-word-lab.csv')

relator_words = {
    'r^3': 'rrr',
    's^2': 'ss',
    'srsr': 'srsr',
}
presentation_check = {
    'relators_evaluate_to_identity': {label: eval_word(word) == E for label, word in relator_words.items()},
    'number_of_distinct_elements_reached': len(shortest),
    'all_S3_elements_reached': set(shortest) == set(S3),
    'max_shortest_word_length': max(len(word) for word in shortest.values()),
}
presentation_check_path = save_json(presentation_check, CHECKS / 'presentation-relations-check.json')

candidate_word = 'rsrsR'
candidate_element = eval_word(candidate_word)
print(f'{candidate_word} evaluates to {name_of[candidate_element]} with permutation {candidate_element}')
display_artifact(presentation_check_path)
pd.DataFrame(presentation_rows).sort_values(['word_length', 'element'])


## 5. Group Actions and the Orbit-Stabilizer Scaffold

A group action is a way for abstract group elements to move points of a set or space. For a left action, the identity fixes every point, and multiplying group elements agrees with doing the motions in sequence. In topology, actions are a bridge between algebra and quotient spaces: an orbit space identifies points that can be moved into one another by the group.

The triangle symmetry group acts on the three vertices. Pick vertex `1`. Its orbit is the set of vertices reachable from it; here that is all three vertices. Its stabilizer is the subgroup of symmetries that leave vertex `1` fixed; here it contains the identity and the reflection through vertex `1`. The orbit-stabilizer equation says that the group is partitioned into equal-sized fibers over the orbit. In this model, six symmetries split into three fibers of two symmetries each.

This is the finite preview of a pattern that becomes geometric later. When a group acts on a topological space, an orbit is a point of the quotient space, and stabilizers measure leftover symmetry at that point. Free actions have trivial stabilizers; nonfree actions have points with symmetry still attached. Covering spaces, projective spaces, and many quotient manifolds are easiest to understand when the action and its stabilizers are visible.


In [ ]:
base_vertex = 0
orbit_vertices = sorted({g[base_vertex] for g in S3})
stabilizer = {g for g in S3 if g[base_vertex] == base_vertex}
fibers = {v: sorted([g for g in S3 if g[base_vertex] == v], key=cycle_name) for v in orbit_vertices}

action_rows = []
for v, members in fibers.items():
    for g in members:
        action_rows.append({'target_vertex': v + 1, 'group_element': name_of[g], 'sends_1_to': g[base_vertex] + 1, 'in_stabilizer_of_1': g in stabilizer})
action_table_path = save_csv(action_rows, TABLES / 'orbit-stabilizer-fibers.csv')
action_check = {
    'group_order': len(S3),
    'orbit_of_vertex_1': [v + 1 for v in orbit_vertices],
    'orbit_size': len(orbit_vertices),
    'stabilizer_of_vertex_1': sorted(name_of[g] for g in stabilizer),
    'stabilizer_size': len(stabilizer),
    'orbit_stabilizer_identity': len(S3) == len(orbit_vertices) * len(stabilizer),
    'fiber_sizes': {str(v + 1): len(members) for v, members in fibers.items()},
}
action_check_path = save_json(action_check, CHECKS / 'action-orbit-stabilizer.json')

fig, ax = plt.subplots(figsize=(8.2, 5.4))
triangle_coords = np.array([[0.12, 0.7], [0.04, 0.35], [0.2, 0.35]])
closed = np.vstack([triangle_coords, triangle_coords[0]])
ax.plot(closed[:, 0], closed[:, 1], color='#374151', linewidth=2)
for idx, (x0, y0) in enumerate(triangle_coords):
    ax.scatter([x0], [y0], s=620, color='#fef3c7', edgecolor='#333333', zorder=3)
    ax.text(x0, y0, str(idx + 1), ha='center', va='center', fontsize=13, weight='bold')
ax.text(0.12, 0.18, 'triangle vertices', ha='center', fontsize=11, weight='bold')

x_columns = {0: 0.45, 1: 0.66, 2: 0.87}
for v, members in fibers.items():
    xcol = x_columns[v]
    ax.scatter([xcol], [0.82], s=780, color='#dbeafe', edgecolor='#333333', zorder=3)
    ax.text(xcol, 0.82, f'1 -> {v + 1}', ha='center', va='center', fontsize=11, weight='bold')
    ax.text(xcol, 0.72, 'fiber', ha='center', fontsize=9, color='#555555')
    for j, g in enumerate(members):
        y0 = 0.58 - 0.13 * j
        color = '#bbf7d0' if g in stabilizer else '#f3f4f6'
        ax.scatter([xcol], [y0], s=680, color=color, edgecolor='#333333', zorder=3)
        ax.text(xcol, y0, name_of[g], ha='center', va='center', fontsize=10)
        ax.plot([triangle_coords[0, 0] + 0.03, xcol - 0.04], [triangle_coords[0, 1], 0.82], color='#9ca3af', linewidth=0.9)

ax.text(0.66, 0.11, '|S3| = 6 = |orbit(1)| 3 times |stabilizer(1)| 2', ha='center', fontsize=12, weight='bold')
ax.text(0.45, 0.245, 'green fiber is the stabilizer of vertex 1', ha='center', fontsize=9, color='#166534')
ax.set_xlim(0, 1)
ax.set_ylim(0.05, 0.95)
ax.set_axis_off()
ax.set_title('Orbit-stabilizer fibers for the action of S3 on triangle vertices')
action_fig_path = save_matplotlib(fig, FIGURES / 'orbit-stabilizer-triangle-action.png')
plt.close(fig)

display_artifact(action_fig_path, width=760)
display_artifact(action_check_path)
pd.DataFrame(action_rows)


## Applied Lab: Diagnose a Quotient Proposal

The most common quotient mistake is trying to multiply cosets by a subgroup that is not normal. The formula looks plausible, but it is not well-defined. We can detect the failure by checking every possible representative. In `S_3`, the subgroup generated by a reflection has two elements and is not normal. Its left cosets still partition the group, but multiplying cosets by choosing representatives produces ambiguous answers.

This lab contrasts that bad quotient proposal with the good quotient by `A_3`. The check is the same in both cases: for every pair of cosets, compute all products of representatives and ask whether the resulting coset label is unique. Normality is exactly the condition that turns this representative check from a gamble into a theorem.


In [ ]:
def subgroup_generated_by(generator):
    return generated_group([generator])


def left_cosets(subgroup):
    remaining = set(S3)
    coset_list = []
    while remaining:
        rep = sorted(remaining, key=cycle_name)[0]
        coset = {compose(rep, h) for h in subgroup}
        coset_list.append((name_of[rep] + 'H', coset))
        remaining -= coset
    return coset_list


def find_coset_name(g, coset_list):
    for label, coset in coset_list:
        if g in coset:
            return label
    raise KeyError(g)


def coset_product_ambiguities(subgroup):
    coset_list = left_cosets(subgroup)
    ambiguous = []
    for left_label, left_coset in coset_list:
        for right_label, right_coset in coset_list:
            outcomes = sorted({find_coset_name(compose(a, b), coset_list) for a in left_coset for b in right_coset})
            if len(outcomes) > 1:
                ambiguous.append({'left': left_label, 'right': right_label, 'possible_products': ', '.join(outcomes)})
    return coset_list, ambiguous


reflection_subgroup = subgroup_generated_by(s)
bad_cosets, bad_ambiguities = coset_product_ambiguities(reflection_subgroup)
good_cosets, good_ambiguities = coset_product_ambiguities(A3)

lab_check = {
    'reflection_subgroup': sorted(name_of[g] for g in reflection_subgroup),
    'reflection_subgroup_normal': all(compose(compose(g, h), inverse(g)) in reflection_subgroup for g in S3 for h in reflection_subgroup),
    'reflection_quotient_ambiguous_product_count': len(bad_ambiguities),
    'A3_normal': normal_A3,
    'A3_quotient_ambiguous_product_count': len(good_ambiguities),
}
lab_check_path = save_json(lab_check, CHECKS / 'non-normal-coset-lab-check.json')
ambiguity_path = save_csv(bad_ambiguities, TABLES / 'non-normal-coset-ambiguities.csv')

display_artifact(lab_check_path)
pd.DataFrame(bad_ambiguities)


## Final Sanity Checks

The final cell verifies both notebook artifacts and algebraic invariants. The artifact checks make sure the generated files exist and are nonempty. The image check guards against blank PNGs. The algebra checks confirm the group axioms for the finite model, the presentation relations, the normal quotient, the first-isomorphism model, the product isomorphism `C_2 x C_3 ~= C_6`, and the orbit-stabilizer identity for the action on vertices.


In [ ]:
expected_artifacts = [
    storyboard_path,
    group_check_path,
    multiplication_table_path,
    cayley_path,
    quotient_table_path,
    quotient_check_path,
    quotient_fig_path,
    proof_path,
    product_check_path,
    product_html_path,
    presentation_table_path,
    presentation_check_path,
    action_table_path,
    action_check_path,
    action_fig_path,
    lab_check_path,
    ambiguity_path,
]
assert_artifacts(expected_artifacts, min_bytes=64)

png_stats = [image_stats(path) for path in [cayley_path, quotient_fig_path, proof_path, action_fig_path]]
assert all(stat['width'] >= 500 and stat['height'] >= 300 for stat in png_stats)
assert all(stat['max_channel_stddev'] > 5 for stat in png_stats)

assert all([closure_ok, associative_ok, identity_ok, inverse_ok])
assert all(relations_ok.values())
assert quotient_check['first_isomorphism_model_holds']
assert product_check['crt_map_bijective'] and product_check['crt_map_homomorphism']
assert product_check['element_(1,1)_has_order_6']
assert all(presentation_check['relators_evaluate_to_identity'].values())
assert action_check['orbit_stabilizer_identity']
assert lab_check['reflection_quotient_ambiguous_product_count'] > 0
assert lab_check['A3_quotient_ambiguous_product_count'] == 0

final_sanity = {
    'artifact_count': len(expected_artifacts),
    'png_stats': png_stats,
    'group_axioms_pass': all([closure_ok, associative_ok, identity_ok, inverse_ok]),
    'presentation_relations_pass': all(relations_ok.values()) and all(presentation_check['relators_evaluate_to_identity'].values()),
    'quotient_and_first_isomorphism_pass': quotient_check['first_isomorphism_model_holds'],
    'product_crt_pass': product_check['crt_map_bijective'] and product_check['crt_map_homomorphism'],
    'orbit_stabilizer_pass': action_check['orbit_stabilizer_identity'],
    'non_normal_lab_detected_failure': lab_check['reflection_quotient_ambiguous_product_count'] > 0,
}
final_sanity_path = save_json(final_sanity, CHECKS / 'final-sanity.json')
assert_artifacts([final_sanity_path], min_bytes=64)
display_artifact(final_sanity_path)
final_sanity


## Takeaways

A group is not just a set with symbols; it is a multiplication system whose identity, inverses, and associativity can be tested in finite models. Generated subgroups are closures under multiplication and inverse, so a Cayley graph is often the cleanest way to see generation.

A homomorphism preserves multiplication, and its kernel records what the map collapses. Kernels are normal, and normality is the exact condition needed for coset multiplication to define a quotient group. The first isomorphism theorem says that any homomorphism is a quotient by its kernel followed by an isomorphism onto its image.

Products multiply coordinate by coordinate, but products can still hide simpler cyclic structure when arithmetic permits it. Direct sums are the finitely supported version used by many algebraic-topology constructions.

Presentations turn generators and relations into a portable description of a group. They are central later because fundamental groups are often computed by naming loop generators and reading relations from topology.

Group actions connect algebra to geometry. Orbits identify points under a symmetry, stabilizers measure the symmetry left at a point, and the orbit-stabilizer count is the finite version of a principle that reappears in quotient spaces and covering transformations.
